In [1]:
import torch
import torch.nn as nn
from tqdm import tqdm
from annoy import AnnoyIndex
import numpy as np

In [2]:
# ADDED
from pathlib import Path
import os
cwd = Path.cwd()
while cwd.name != "PyTorchNLPBook":
    if cwd.parent == cwd: raise RutimeError("Couldn't find PyTorchNLPBook")
    cwd = cwd.parent
os.chdir(cwd)
Path.cwd()

WindowsPath('D:/Source/gh-pc/PyTorchNLPBook')

#### Comment re GloVe
The Glove data file was downloaded from <https://huggingface.co/datasets/SLU-CSCI4750/glove.6B.100d.txt/tree/main> and copied into the `/data/glove/` folder. However, it's too large to update to GitHub, and so the `/data/glove/` folder has been added to `.gitignore`.

#### Comments (added)
The `Annoy` module ("[Approximate Nearest Neighbor, Oh Yeah!](https://github.com/spotify/annoy)") is used to create an index for searching. The vectors encpasulate similairities, but a brute force NN search given a word would be O(N) (where N = |vocab|). The annoy index uses trees to manage scale of searches. Given a query word and a certain similar word, searching in a single tree won't always find that similar word. The `annoy.build()` argument `n_trees` determines the number of trees created. More trees take longer to build, but provide more opportunity that similar words will be found.

In [17]:
class PreTrainedEmbeddings(object):
    """ A wrapper around pre-trained word vectors and their use """
    def __init__(self, word_to_index, word_vectors):
        """
        Args:
            word_to_index (dict): mapping from word to integers
            word_vectors (list of numpy arrays)
        """
        self.word_to_index = word_to_index
        self.word_vectors = word_vectors
        self.index_to_word = {v: k for k, v in self.word_to_index.items()}

        self.index = AnnoyIndex(len(word_vectors[0]), metric='euclidean')
        print("Building Index!")
        for _, i in self.word_to_index.items():
            self.index.add_item(i, self.word_vectors[i])
        self.index.build(50)
        print("Finished!")
        
    @classmethod
    def from_embeddings_file(cls, embedding_file):
        """Instantiate from pre-trained vector file.
        
        Vector file should be of the format:
            word0 x0_0 x0_1 x0_2 x0_3 ... x0_N
            word1 x1_0 x1_1 x1_2 x1_3 ... x1_N
        
        Args:
            embedding_file (str): location of the file
        Returns: 
            instance of PretrainedEmbeddigns
        """
        word_to_index = {}
        word_vectors = []

        with open(embedding_file, encoding="utf-8") as fp:
            for line in fp.readlines():
                line = line.split(" ")
                word = line[0]
                vec = np.array([float(x) for x in line[1:]])
                
                word_to_index[word] = len(word_to_index)
                word_vectors.append(vec)
                
        return cls(word_to_index, word_vectors)
    
    def get_embedding(self, word):
        """
        Args:
            word (str)
        Returns
            an embedding (numpy.ndarray)
        """
        return self.word_vectors[self.word_to_index[word]]

    def get_closest_to_vector(self, vector, n=1):
        """Given a vector, return its n nearest neighbors
        
        Args:
            vector (np.ndarray): should match the size of the vectors 
                in the Annoy index
            n (int): the number of neighbors to return
        Returns:
            [str, str, ...]: words that are nearest to the given vector. 
                The words are not ordered by distance 
        """
        nn_indices = self.index.get_nns_by_vector(vector, n)
        return [self.index_to_word[neighbor] for neighbor in nn_indices]
    
    def compute_and_print_analogy(self, word1, word2, word3, search_count=4):
        """Prints the solutions to analogies using word embeddings

        Analogies are word1 is to word2 as word3 is to __
        This method will print: word1 : word2 :: word3 : word4
        
        Args:
            word1 (str)
            word2 (str)
            word3 (str)
        """
        vec1 = self.get_embedding(word1)
        vec2 = self.get_embedding(word2)
        vec3 = self.get_embedding(word3)

        # now compute the fourth word's embedding!
        spatial_relationship = vec2 - vec1
        vec4 = vec3 + spatial_relationship

        closest_words = self.get_closest_to_vector(vec4, n=search_count)
        existing_words = set([word1, word2, word3])
        closest_words = [word for word in closest_words 
                             if word not in existing_words] 

        if len(closest_words) == 0:
            print("Could not find nearest neighbors for the computed vector!")
            return
        
        for word4 in closest_words:
            print("{} : {} :: {} : {}".format(word1, word2, word3, word4))

In [18]:
embeddings = PreTrainedEmbeddings.from_embeddings_file('data/glove/glove.6B.100d.txt')

Building Index!
Finished!


In [20]:
embeddings.compute_and_print_analogy('man', 'he', 'woman')

man : he :: woman : she
man : he :: woman : her
man : he :: woman : having


In [21]:
embeddings.compute_and_print_analogy('fly', 'plane', 'sail')

fly : plane :: sail : ship
fly : plane :: sail : vessel


In [22]:
embeddings.compute_and_print_analogy('cat', 'kitten', 'dog')

cat : kitten :: dog : puppy
cat : kitten :: dog : bulldog
cat : kitten :: dog : mannequin


In [23]:
embeddings.compute_and_print_analogy('blue', 'color', 'dog')

blue : color :: dog : breed
blue : color :: dog : pig
blue : color :: dog : bites
blue : color :: dog : personality


In [24]:
embeddings.compute_and_print_analogy('leg', 'legs', 'hand')

leg : legs :: hand : hands
leg : legs :: hand : ears
leg : legs :: hand : eyes


In [27]:
embeddings.compute_and_print_analogy('toe', 'foot', 'finger')

toe : foot :: finger : ground
toe : foot :: finger : pointing
toe : foot :: finger : moving


In [32]:
embeddings.compute_and_print_analogy('toe', 'foot', 'finger',search_count=6)

toe : foot :: finger : hand
toe : foot :: finger : kept
toe : foot :: finger : fingers
toe : foot :: finger : ground


In [33]:
embeddings.compute_and_print_analogy('talk', 'communicate', 'read')

talk : communicate :: read : interpret
talk : communicate :: read : memorize
talk : communicate :: read : communicated


In [34]:
embeddings.compute_and_print_analogy('blue', 'democrat', 'red')

blue : democrat :: red : republican
blue : democrat :: red : congressman
blue : democrat :: red : senator


In [35]:
embeddings.compute_and_print_analogy('man', 'king', 'woman')

man : king :: woman : queen
man : king :: woman : monarch
man : king :: woman : throne


In [36]:
embeddings.compute_and_print_analogy('man', 'doctor', 'woman')

man : doctor :: woman : nurse
man : doctor :: woman : physician


In [39]:
embeddings.compute_and_print_analogy('fast', 'fastest', 'small')

fast : fastest :: small : smallest
fast : fastest :: small : largest
fast : fastest :: small : quarters
